In [0]:
import pyspark.sql.functions as F
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import plotly.express as px



Base de Machine Learning integrada com sucesso! Total de 18360 usuários.


caminho,qtd_touchpoints,inicio_google,tem_acesso_direto,churn_funil
(data deleted) > (data deleted) > (data deleted),3,0,0,1
(data deleted) > (data deleted) > (data deleted) > > > > > > > > > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > shop.googlemerchandisestore.com > shop.googlemerchandisestore.com,50,0,0,1
(data deleted) > (data deleted) > (data deleted) > google > google > google > google,7,0,0,1
(data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google > google,32,0,0,1
(data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted) > (data deleted),20,0,0,1


In [0]:
# 1. Carregar a tabela bruta direto do catálogo do Databricks
nome_tabela = "workspace.default.bq_results_20260617_234748_1781740105311" 
df_ga4 = spark.table(nome_tabela)

# 2. Criar a flag de compra e filtrar registros sem origem
df_com_conversao = df_ga4.withColumn(
    "comprou",
    F.when(F.col("event_name") == "purchase", 1).otherwise(0)
).filter(F.col("utm_source").isNotNull())

# 3. Agrupar a jornada completa por ID de usuário
df_jornada_final = df_com_conversao.groupBy("user_pseudo_id").agg(
    F.array_join(F.collect_list("utm_source"), " > ").alias("caminho"),
    F.max("comprou").alias("conversao")
)

# 4. Trazer os dados agregados do Spark para o Pandas local
df_ml = df_jornada_final.toPandas()

# 5. Criar as variáveis preditivas com base no comportamento de navegação
# Conta quantos canais o usuário visitou na jornada
df_ml['qtd_touchpoints'] = df_ml['caminho'].apply(lambda x: len(str(x).split('>')))

# Flag se a primeira visita começou pelo Google
df_ml['inicio_google'] = df_ml['caminho'].apply(lambda x: 1 if 'google' in str(x).lower().split('>')[0] else 0)

# Flag se o usuário passou pelo canal direto em algum momento
df_ml['tem_acesso_direto'] = df_ml['caminho'].apply(lambda x: 1 if 'direct' in str(x).lower() else 0)

# 6. Definir a Variável Alvo (Se conversão é 0, o usuário abandonou o funil = Churn)
df_ml['churn_funil'] = (df_ml['conversao'] == 0).astype(int)

print(f"Base de Machine Learning integrada com sucesso! Total de {len(df_ml)} usuários.")
display(df_ml[['caminho', 'qtd_touchpoints', 'inicio_google', 'tem_acesso_direto', 'churn_funil']].head())

In [0]:
# Definindo X e y
features = ['qtd_touchpoints', 'inicio_google', 'tem_acesso_direto']
X = df_ml[features]
y = df_ml['churn_funil']

# Padronizando a escala para o algoritmo não enviesar os pesos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Separando 30% da base para teste cego
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# Treinando a Regressão Logística com balanceamento de classes
modelo = LogisticRegression(class_weight='balanced')
modelo.fit(X_train, y_train)

# Avaliando a capacidade preditiva
auc = roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1])
print(f"Qualidade de Separação (ROC-AUC): {auc:.3f}")

Qualidade de Separação (ROC-AUC): 0.983


In [0]:
# Extraindo os pesos matemáticos do modelo
df_coef = pd.DataFrame({
    'Variavel': features,
    'Peso': modelo.coef_[0]
}).sort_values(by='Peso', ascending=True)

# Vermelho = Aumenta a chance do cliente ir embora / Azul = Ajuda a reter e converter
df_coef['Cor'] = np.where(df_coef['Peso'] > 0, '#ef233c', '#00b4d8')

fig1 = px.bar(
    df_coef, 
    x='Peso', 
    y='Variavel', 
    orientation='h',
    title="Causas de Churn de Funil (Regressão Logística)",
    text='Peso',
    color='Cor',
    color_discrete_map="identity"
)

fig1.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig1.update_layout(template='plotly_dark', showlegend=False, xaxis_title="Impacto no Abandono (Log-Odds)", yaxis_title="")
fig1.show()

In [0]:
# Aplicando a probabilidade de 0% a 100% para cada usuário na base
df_ml['prob_abandono_%'] = (modelo.predict_proba(X_scaled)[:, 1] * 100).round(2)

# Plotando a distribuição de risco
fig2 = px.histogram(
    df_ml, 
    x='prob_abandono_%', 
    nbins=20,
    title="Risco de Abandono de Carrinho / Funil",
    color_discrete_sequence=['#8d99ae']
)
fig2.add_vline(x=75, line_dash="dash", line_color="#ef233c", annotation_text="Alta Propensão a Churn")
fig2.update_layout(template='plotly_dark', xaxis_title="Probabilidade de Abandono (%)", yaxis_title="Volume de Usuários")
fig2.show()

# Gerando a base para ação de marketing (Remarketing / Oferta Futura)
base_remarketing = df_ml[df_ml['prob_abandono_%'] > 75].sort_values(by='prob_abandono_%', ascending=False)
print(f"\nLista de Ação gerada: {len(base_remarketing)} leads com alta propensão a abandonar o funil.")

# Essa é a tabela que a Orbital vai amar ver: o usuário e a probabilidade dele sair
display(base_remarketing[['user_pseudo_id', 'prob_abandono_%', 'caminho']].head(10))


Lista de Ação gerada: 16648 leads com alta propensão a abandonar o funil.


user_pseudo_id,prob_abandono_%,caminho
3.618763213190718E7,98.52,(data deleted)
1.9547218710262243E7,98.52,(data deleted)
6015650.401212364,98.52,(data deleted)
5977954.137613164,98.52,(data deleted)
8452593.746369645,98.52,(data deleted)
4.8997196260011256E7,98.37,(data deleted) > (data deleted)
1971033.4844883706,98.37,(data deleted) > (data deleted)
6.092447038415345E7,98.37,(data deleted) > (data deleted)
5.8191284935696855E7,98.37,(data deleted) > (data deleted)
3.927007463449546E7,98.37,(data deleted) > (data deleted)
